# 01 - Export ProLLaMA and Mol-LLaMA to ONNX 🚀

Run this notebook in a Colab TPU runtime with a high-memory host. Export runs on the TPU VM CPU; TPU cores are not used.

The notebook exports INT8 encoder graphs, checks PyTorch/ONNX parity, and uploads each encoder to a separate Hugging Face model repository.

In [ ]:
%cd /content
![ -d Protein-Ligand-Affinity-Prediction-Using-LLM ] || git clone https://github.com/karthikeyanr103/Protein-Ligand-Affinity-Prediction-Using-LLM.git
%cd /content/Protein-Ligand-Affinity-Prediction-Using-LLM
!git pull
%pip install -U pip setuptools wheel ninja packaging
%pip install -e ".[export]"
!mkdir -p external
![ -d external/Mol-LLaMA ] || git clone --depth 1 https://github.com/DongkiKim95/Mol-LLaMA external/Mol-LLaMA
!sed -e 's/rdkit-pypi/rdkit/g' -e '/^torch==/d' -e '/^torchvision==/d' -e '/^torchaudio==/d' -e '/^torch-geometric/d' -e '/^torch_scatter/d' -e '/^torch-scatter/d' -e '/^peft==/d' external/Mol-LLaMA/requirements.txt > /content/mol_llama_requirements.txt
%pip install -r /content/mol_llama_requirements.txt
%pip install "peft>=0.17,<1" openbabel-wheel ogb torch-geometric

In [ ]:
import subprocess, sys, torch
version = '.'.join(torch.__version__.split('+')[0].split('.')[:2]) + '.0'
tag = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
url = f'https://data.pyg.org/whl/torch-{version}+{tag}.html'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', url])
print('Restart the Colab session after this cell, then continue.')

## Configuration

In [ ]:
%cd /content/Protein-Ligand-Affinity-Prediction-Using-LLM
from google.colab import drive, userdata
from huggingface_hub import HfApi, login
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
HF_USER = 'your-huggingface-username'
PRO_REPO = f'{HF_USER}/prollama-affinity-onnx'
MOL_REPO = f'{HF_USER}/mol-llama-affinity-onnx'
ROOT = Path('/content/drive/MyDrive/protein_affinity/onnx')
PRO_ONNX = ROOT / 'prollama'
MOL_ONNX = ROOT / 'mol_llama'
PRO_ONNX.mkdir(parents=True, exist_ok=True)
MOL_ONNX.mkdir(parents=True, exist_ok=True)
token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = token
login(token=token)

## Export ProLLaMA

In [ ]:
if not (PRO_ONNX / 'prollama_encoder_int8.onnx').exists():
    subprocess.run([
        'affinity-export-protein-onnx',
        '--model-id', 'GreatCaptainNemo/ProLLaMA',
        '--output', str(PRO_ONNX),
        '--dtype', 'float32',
        '--sequence-length', '32',
        '--quantize',
    ], check=True)
else:
    print('ProLLaMA export already exists.')

## Export Mol-LLaMA molecular encoder

In [ ]:
if not (MOL_ONNX / 'mol_llama_encoder_int8.onnx').exists():
    subprocess.run([
        'affinity-export-mol-onnx',
        '--official-repo', 'external/Mol-LLaMA',
        '--model-id', 'DongkiKim/Mol-Llama-3.1-8B-Instruct',
        '--output', str(MOL_ONNX),
        '--quantize',
    ], check=True)
else:
    print('Mol-LLaMA export already exists.')

## ONNX smoke test

In [ ]:
from affinity.onnx_embeddings import ProLLaMAOnnxEmbedder, MolLLaMAOnnxEmbedder
protein = ProLLaMAOnnxEmbedder(PRO_ONNX).encode(['ACDEFGHIKLMNPQRSTVWY'])
molecule = MolLLaMAOnnxEmbedder(MOL_ONNX).encode(['CC(=O)O'])
print('Protein embedding:', protein.shape)
print('Molecule embedding:', molecule.shape)

## Upload both encoder repositories

In [ ]:
UPLOAD = False
if UPLOAD:
    api = HfApi(token=token)
    for repo_id, folder in [(PRO_REPO, PRO_ONNX), (MOL_REPO, MOL_ONNX)]:
        api.create_repo(repo_id, repo_type='model', exist_ok=True)
        api.upload_large_folder(repo_id=repo_id, repo_type='model', folder_path=str(folder))
        print('Uploaded:', repo_id)